## Auditoria final

Esse notebook novo (`02_silver_quality_check.py`) serve como uma **auditoria** final da camada Silver inteira: pra cada uma das 6 tabelas que processamos, comparamos:

- Quantas linhas tinha na Bronze vs. quantas sobraram na Silver (quanto foi descartado, e se essa quantidade faz sentido)
- Quantos nulos existiam nas colunas-chave antes vs. depois (deve ir a zero depois)
- Quantas duplicatas existiam antes vs. depois (idem)

**Por que isso é importante**

1. Detecção de regressão - se algum dia a gente alterar a lógica de limpeza (numa Sprint futura, ou revisando o código), esse notebook funciona como um "teste automatizado informal": rodar ele de novo revela na hora se alguma tabela quebrou o padrão esperado.

2. Evidência auditável, num lugar só - em vez de vasculhar 4 PRs diferentes procurando "cadê a prova de que orders ficou limpo", você tem um relatório único, consolidado, que qualquer pessoa (ou você mesma, meses depois) consulta pra confirmar a qualidade de toda a camada de uma vez.

3. Antes de avançar pra Gold, você quer ter certeza - a camada Gold vai fazer JOINs entre essas tabelas. Se alguma tiver um problema de qualidade não detectado, ele se propaga e vira um bug muito mais difícil de rastrear lá na frente. Esse notebook é o "portão de qualidade" antes de seguir adiante.

### 1. Validação de qualidade entre as camadas Bronze e Silver do Lakehouse

In [0]:
%python
tabelas = ["orders", "customers", "products", "sellers", "order_items", "order_payments"]

resumo = []

for t in tabelas:
    bronze_count = spark.table(f"olist_project.bronze.{t}").count()
    silver_count = spark.table(f"olist_project.silver.{t}").count()
    descartadas = bronze_count - silver_count
    pct_descartado = round((descartadas / bronze_count) * 100, 2) if bronze_count > 0 else 0

    resumo.append({
        "tabela": t,
        "bronze_linhas": bronze_count,
        "silver_linhas": silver_count,
        "linhas_descartadas": descartadas,
        "pct_descartado": pct_descartado,
    })

resumo_df = spark.createDataFrame(resumo)
resumo_df.createOrReplaceTempView("resumo_qualidade")

display(resumo_df)